<a href="https://colab.research.google.com/github/USHODAYAKALYANI/LSTM-Prediction/blob/main/trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import tensorflow as tf

In [2]:
train_df = pd.read_excel('/content/train.xlsx')
val_df = pd.read_excel('/content/val.xlsx')
test_df = pd.read_excel('/content/test.xlsx')

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(302374, 14)
(81605, 14)
(81606, 14)


In [3]:
features = ['Speed','Ax','Ay','Az','Gx','Gy','Gz']

In [4]:
X_train = train_df[features]
y_train = train_df['Label']

In [5]:
X_val = val_df[features]
y_val = val_df['Label']

In [6]:
X_test = test_df[features]
y_test = test_df['Label']

In [7]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)

y_val = le.transform(y_val)

y_test = le.transform(y_test)

print(le.classes_)

['BUMP' 'LEFT' 'RIGHT' 'STOP' 'STRAIGHT']


In [8]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

In [9]:
window = 20

In [23]:
from collections import Counter

def create_sequences(X, y, window):

    X_seq = []
    y_seq = []

    for i in range(len(X)-window):

        X_seq.append(X[i:i+window])

        counts = Counter(y[i:i+window])

        majority_label = counts.most_common(1)[0][0]

        y_seq.append(majority_label)

    return np.array(X_seq), np.array(y_seq)

In [24]:
window = 20

X_train_seq, y_train_seq = create_sequences(X_train, y_train, window)

X_val_seq, y_val_seq = create_sequences(X_val, y_val, window)

X_test_seq, y_test_seq = create_sequences(X_test, y_test, window)

In [25]:
print(X_train_seq.shape)
print(y_train_seq.shape)

print(X_val_seq.shape)
print(y_val_seq.shape)

print(X_test_seq.shape)
print(y_test_seq.shape)

(302354, 20, 7)
(302354,)
(81585, 20, 7)
(81585,)
(81586, 20, 7)
(81586,)


In [26]:
import tensorflow as tf

model = tf.keras.Sequential([

    tf.keras.layers.LSTM(
        64,
        input_shape=(20,7)
    ),

    tf.keras.layers.Dense(
        32,
        activation='relu'
    ),

    tf.keras.layers.Dense(
        len(le.classes_),
        activation='softmax'
    )
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,677 (80.77 KB)

 Trainable params: 20,677 (80.77 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history = model.fit(
    X_train_seq,
    y_train_seq,
    epochs=10,
    batch_size=64,
    validation_data=(X_val_seq, y_val_seq)
)

Epoch 1/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 67s 14ms/step - accuracy: 0.8118 - loss: 0.5691 - val_accuracy: 0.8100 - val_loss: 0.6265
Epoch 2/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 69s 15ms/step - accuracy: 0.8322 - loss: 0.4906 - val_accuracy: 0.7830 - val_loss: 0.6760
Epoch 3/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 77s 14ms/step - accuracy: 0.8381 - loss: 0.4629 - val_accuracy: 0.8101 - val_loss: 0.5766
Epoch 4/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 64s 14ms/step - accuracy: 0.8476 - loss: 0.4395 - val_accuracy: 0.8169 - val_loss: 0.5720
Epoch 5/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 83s 14ms/step - accuracy: 0.8522 - loss: 0.4259 - val_accuracy: 0.8159 - val_loss: 0.5689
Epoch 6/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 65s 14ms/step - accuracy: 0.8574 - loss: 0.4098 - val_accuracy: 0.8217 - val_loss: 0.5465
Epoch 7/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 66s 14ms/step - accuracy: 0.8611 - loss: 0.3949 - val_accuracy: 0.8197 - val_loss: 0.5615
Epoch 8/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 81s 14ms/step - accuracy: 0.8654 -

In [28]:
loss, accuracy = model.evaluate(
    X_val_seq,
    y_val_seq
)

print("Validation Accuracy:", accuracy)
print("Validation Loss:", loss)

2550/2550 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.8233 - loss: 0.5534
Validation Accuracy: 0.8232518434524536
Validation Loss: 0.5533820986747742


In [29]:
loss, accuracy = model.evaluate(
    X_test_seq,
    y_test_seq
)

print("Test Accuracy:", accuracy)
print("Test Loss:", loss)

2550/2550 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.6272 - loss: 1.2562
Test Accuracy: 0.6271909475326538
Test Loss: 1.2561556100845337


In [30]:
y_pred = model.predict(X_test_seq)

y_pred_classes = np.argmax(y_pred, axis=1)

from sklearn.metrics import classification_report

print(classification_report(
    y_test_seq,
    y_pred_classes
))

2550/2550 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step
              precision    recall  f1-score   support

           0       0.56      0.20      0.30     18030
           1       0.62      0.50      0.55      8897
           2       0.57      0.31      0.40      4864
           3       0.95      0.71      0.81     14735
           4       0.57      0.89      0.70     35060

    accuracy                           0.63     81586
   macro avg       0.66      0.52      0.55     81586
weighted avg       0.64      0.63      0.60     81586

